# Titanic Survival Prediction

This notebook is a beginner-friendly introduction to Kaggle Titanic competition using a simple rule-based model.

## Objective

In this notebook, I analyze the Titanic dataset to understand which factors affect survival, and build a simple rule-based model to predict whether a passenger survived or not.

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


In [2]:
import pandas as pd

train = pd.read_csv('/kaggle/input/competitions/titanic/train.csv')
train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
test = pd.read_csv('/kaggle/input/competitions/titanic/test.csv')

test['Survived'] = 0

submission = test[['PassengerId', 'Survived']]
submission.to_csv('submission.csv', index=False)

In [4]:
test['Survived'] = (test['Sex'] == 'female').astype(int)

In [5]:
test[['Sex', 'Survived']].head(10)

,Sex,Survived
0,male,0
1,female,1
2,male,0
3,male,0
4,female,1
5,male,0
6,female,1
7,male,0
8,female,1
9,male,0


In [6]:
test['Survived'] = (
    (test['Sex'] == 'female') | (test['Pclass'] == 1)
).astype(int)

In [7]:
test['Age'] = test['Age'].fillna(30)
test['Survived'] = (
    (test['Sex'] == 'female') |
    (test['Pclass'] == 1) |
    (test['Age'] < 10)
).astype(int)

In [8]:
# 性別
train.groupby('Sex')['Survived'].mean()

# クラス
train.groupby('Pclass')['Survived'].mean()

# 年齢
train['AgeGroup'] = pd.cut(train['Age'], bins=[0,10,20,30,40,50,60,70,80])
train.groupby('AgeGroup')['Survived'].mean()

/tmp/ipykernel_17/3829429435.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  train.groupby('AgeGroup')['Survived'].mean()


AgeGroup
(0, 10]     0.593750
(10, 20]    0.382609
(20, 30]    0.365217
(30, 40]    0.445161
(40, 50]    0.383721
(50, 60]    0.404762
(60, 70]    0.235294
(70, 80]    0.200000
Name: Survived, dtype: float64

## Exploratory Data Analysis

From the analysis above:

- Females have a much higher survival rate than males.
- Passengers in higher classes (especially Pclass 1) have higher survival rates.
- Young children (Age < 10) tend to have higher survival rates.

These observations suggest that gender, passenger class, and age are important factors in predicting survival.

## Hypothesis

Based on the analysis above, I make the following assumptions:

- Female passengers are more likely to survive.
- Passengers in higher classes (especially Pclass 1) are more likely to survive.
- Young children (Age < 10) are more likely to survive.

These factors will be used to build a simple rule-based prediction model.

## Model

Based on the hypotheses, I construct a simple rule-based model.

A passenger is predicted to survive if:

- The passenger is female, OR
- The passenger is in Pclass 1, OR
- The passenger is a child (Age < 10)

In [9]:
submission = test[['PassengerId', 'Survived']]
submission.to_csv('submission.csv', index=False)

In [10]:
# ===== ① 欠損値の確認 =====
print("Ageの欠損数:", test['Age'].isnull().sum())

# ===== ② 欠損値の処理 =====
test['Age'] = test['Age'].fillna(30)

Ageの欠損数: 0


In [11]:
submission = test[['PassengerId', 'Survived']]
submission.to_csv('submission.csv', index=False)

## Result

The model achieved a score of approximately 0.70 on the public leaderboard.

This indicates that gender and passenger class are strong predictors of survival. 
However, the model is still very simple, and performance could likely be improved by using machine learning techniques.